# Hierarchical 멀티 에이전트 — 팀을 지휘하는 최상위 관리자

**Hierarchical(계층)** 구조는 Supervisor 를 **여러 층**으로 쌓는다. 작업자가 너무 많아지면 관리자 한 명이 다 챙기기 어렵다 → **팀(서브그래프)** 으로 묶고, **최상위 supervisor 가 팀을 지휘** 한다.

```
                  teams_supervisor (대장)
                   /              \
          research_team        writing_team        ← 각 팀은 그 자체로 Supervisor 그래프
          /         \          /     |      \
       search   web_scraper  doc  note  chart
```

구성:
- **research_team**(서브그래프): 팀 supervisor + search + web_scraper
- **writing_team**(서브그래프): 팀 supervisor + doc_writer + note_taker + chart_generator
- **teams_supervisor**(최상위): 두 팀을 작업자처럼 호출

핵심: [multi_agent] 04 의 Supervisor 를 **팩토리 함수(`make_supervisor_node`)** 로 만들어 **층마다 재사용** 하고, 팀 서브그래프를 상위 그래프의 노드로 감싼다.

> `OPENAI_API_KEY`, `TAVILY_API_KEY` 필요.

## 환경 변수 준비

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ.setdefault("USER_AGENT", "ai-agent-study")
for k in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    assert os.environ.get(k), f"{k} 가 .env 에 없습니다"
print("환경변수 로드 완료")

## 1. 웹 검색용 도구
[basics] Tavily 검색 + 웹페이지 스크래핑 도구.

In [ ]:
from typing import Annotated, List
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool

tavily_tool = TavilySearchResults(max_results=5)

@tool
def scrape_webpages(urls: List[str]) -> str:
    """Scrape the provided web pages for detailed information."""
    docs = WebBaseLoader(urls).load()
    return "\n\n".join(
        f'<Document name="{d.metadata.get("title", "")}">\n{d.page_content}\n</Document>'
        for d in docs
    )

## 2. 문서 작성용 도구
개요 작성/읽기/쓰기/편집 + 차트 생성 도구. 파일 기반으로 협업한다.

In [ ]:
from typing import Dict, Optional

@tool
def create_outline(
    points: Annotated[List[str], "List of main points or sections."],
    file_name: Annotated[str, "File path to save the outline."],
) -> str:
    """Create and save an outline."""
    with open(file_name, "w", encoding="utf-8") as f:
        for i, point in enumerate(points):
            f.write(f"{i + 1}. {point}\n")
    return f"Outline saved to {file_name}"

@tool
def read_document(
    file_name: Annotated[str, "File path to read."],
    start: Annotated[Optional[int], "Start line (default 0)."] = None,
    end: Annotated[Optional[int], "End line (default None)."] = None,
) -> str:
    """Read the specified document."""
    with open(file_name, "r", encoding="utf-8") as f:
        lines = f.readlines()
    return "\n".join(lines[(start or 0):end])

@tool
def write_document(
    content: Annotated[str, "Text content to write."],
    file_name: Annotated[str, "File path to save."],
) -> str:
    """Create and save a text document."""
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(content)
    return f"Document saved to {file_name}"

@tool
def edit_document(
    file_name: Annotated[str, "Path of the document to edit."],
    inserts: Annotated[Dict[int, str], "line number(1-indexed) -> text to insert"],
) -> str:
    """Edit a document by inserting text at specific line numbers."""
    with open(file_name, "r", encoding="utf-8") as f:
        lines = f.readlines()
    for line_number, text in sorted(inserts.items()):
        if 1 <= line_number <= len(lines) + 1:
            lines.insert(line_number - 1, text + "\n")
        else:
            return f"Error: Line number {line_number} out of range."
    with open(file_name, "w", encoding="utf-8") as f:
        f.writelines(lines)
    return f"Document edited and saved to {file_name}"

@tool
def python_exec_tool(
    code: Annotated[str, "The python code to execute to generate your chart."],
):
    """Execute python code. Print values you want to see with print(...)."""
    try:
        result = exec(code)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"
    return f"Successfully executed:\n{code}\nStdout: {result}"

## 3. Supervisor 팩토리

[multi_agent] 04 의 supervisor 를 **함수로 일반화** 한다. members 리스트만 주면 그 작업자들을 지휘하는 supervisor 노드를 찍어낸다 → 팀마다, 그리고 최상위에서 재사용.

In [ ]:
from typing import Literal
from typing_extensions import TypedDict
from langchain_core.language_models.chat_models import BaseChatModel
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import Command

class State(MessagesState):
    next: str

def make_supervisor_node(llm: BaseChatModel, members: list[str]):
    options = ["FINISH"] + members
    system_prompt = (
        "You are a supervisor managing a conversation between these workers: "
        f"{members}. Given the user request, respond with the worker to act next. "
        "Each worker performs a task and reports results. When finished, respond with FINISH."
    )

    class Router(TypedDict):
        """다음 작업자. 필요 없으면 FINISH."""
        next: Literal[*options]

    def supervisor_node(state: State) -> Command[Literal[*members, "__end__"]]:
        messages = [{"role": "system", "content": system_prompt}] + state["messages"]
        goto = llm.with_structured_output(Router).invoke(messages)["next"]
        if goto == "FINISH":
            goto = END
        return Command(goto=goto, update={"next": goto})

    return supervisor_node

## 4. research_team 서브그래프

팀 supervisor + search + web_scraper. [multi_agent] 각 작업자는 일 끝나면 `goto="supervisor"` 로 **팀 내부 supervisor** 에게 복귀한다 (04 와 동일).

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(model="gpt-4o")

search_agent = create_react_agent(llm, tools=[tavily_tool])
def search_node(state: State) -> Command[Literal["supervisor"]]:
    result = search_agent.invoke(state)
    return Command(
        update={"messages": [HumanMessage(content=result["messages"][-1].content, name="search")]},
        goto="supervisor",
    )

web_scraper_agent = create_react_agent(llm, tools=[scrape_webpages])
def web_scraper_node(state: State) -> Command[Literal["supervisor"]]:
    result = web_scraper_agent.invoke(state)
    return Command(
        update={"messages": [HumanMessage(content=result["messages"][-1].content, name="web_scraper")]},
        goto="supervisor",
    )

In [ ]:
research_supervisor_node = make_supervisor_node(llm, ["search", "web_scraper"])

research_builder = StateGraph(State)
research_builder.add_node("supervisor", research_supervisor_node)
research_builder.add_node("search", search_node)
research_builder.add_node("web_scraper", web_scraper_node)
research_builder.add_edge(START, "supervisor")
research_graph = research_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(research_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(research_graph.get_graph().draw_mermaid())

## 5. writing_team 서브그래프
팀 supervisor + doc_writer + note_taker + chart_generator. 같은 패턴, 작업자만 다르다.

In [ ]:
doc_writer_agent = create_react_agent(
    llm, tools=[write_document, edit_document, read_document],
    prompt="You can read, write and edit documents based on note-taker's outlines. Don't ask follow-up questions.",
)
def doc_writing_node(state: State) -> Command[Literal["supervisor"]]:
    result = doc_writer_agent.invoke(state)
    return Command(
        update={"messages": [HumanMessage(content=result["messages"][-1].content, name="doc_writer")]},
        goto="supervisor",
    )

note_taking_agent = create_react_agent(
    llm, tools=[create_outline, read_document],
    prompt="You can read documents and create outlines for the document writer. Don't ask follow-up questions.",
)
def note_taking_node(state: State) -> Command[Literal["supervisor"]]:
    result = note_taking_agent.invoke(state)
    return Command(
        update={"messages": [HumanMessage(content=result["messages"][-1].content, name="note_taker")]},
        goto="supervisor",
    )

chart_generating_agent = create_react_agent(llm, tools=[read_document, python_exec_tool])
def chart_generating_node(state: State) -> Command[Literal["supervisor"]]:
    result = chart_generating_agent.invoke(state)
    return Command(
        update={"messages": [HumanMessage(content=result["messages"][-1].content, name="chart_generator")]},
        goto="supervisor",
    )

In [ ]:
doc_writing_supervisor_node = make_supervisor_node(
    llm, ["doc_writer", "note_taker", "chart_generator"]
)

paper_writing_builder = StateGraph(State)
paper_writing_builder.add_node("supervisor", doc_writing_supervisor_node)
paper_writing_builder.add_node("doc_writer", doc_writing_node)
paper_writing_builder.add_node("note_taker", note_taking_node)
paper_writing_builder.add_node("chart_generator", chart_generating_node)
paper_writing_builder.add_edge(START, "supervisor")
paper_writing_graph = paper_writing_builder.compile()

## 6. 최상위 supervisor — 팀을 작업자처럼 호출

핵심: **팀 서브그래프를 노드로 감싼다.** `call_research_team` 노드가 `research_graph.invoke(...)` 로 팀 전체를 실행하고, 결과만 최상위로 가져와 다시 `goto="supervisor"` 로 대장에게 복귀한다.

[basics] 04 의 작업자 노드와 모양이 같다 — 다만 "작업자" 자리에 **팀 그래프** 가 들어갈 뿐.

In [ ]:
teams_supervisor_node = make_supervisor_node(llm, ["research_team", "writing_team"])

def call_research_team(state: State) -> Command[Literal["supervisor"]]:
    response = research_graph.invoke({"messages": state["messages"][-1]})
    return Command(
        update={"messages": [HumanMessage(content=response["messages"][-1].content, name="research_team")]},
        goto="supervisor",
    )

def call_paper_writing_team(state: State) -> Command[Literal["supervisor"]]:
    response = paper_writing_graph.invoke({"messages": state["messages"][-1]})
    return Command(
        update={"messages": [HumanMessage(content=response["messages"][-1].content, name="writing_team")]},
        goto="supervisor",
    )

In [ ]:
super_builder = StateGraph(State)
super_builder.add_node("supervisor", teams_supervisor_node)
super_builder.add_node("research_team", call_research_team)
super_builder.add_node("writing_team", call_paper_writing_team)
super_builder.add_edge(START, "supervisor")
super_graph = super_builder.compile()

In [ ]:
try:
    display(Image(super_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(super_graph.get_graph().draw_mermaid())

## 테스트

조사(research_team) → 문서 작성(writing_team) 이 최상위 supervisor 지휘 아래 협업한다.
[multi_agent] `subgraphs=True` 로 팀 내부 진행까지 스트리밍한다.

In [ ]:
for team, chunk in super_graph.stream(
    {"messages": [("user",
        "AI Agent 에 대해 조사한 뒤, 조사가 끝나면 문서 작성 팀으로 'AI Agent 개념 도서'의 "
        "목차(장-절 구성)를 제안하여 텍스트 파일로 저장해주세요.")]},
    {"recursion_limit": 150},
    subgraphs=True,
    stream_mode="updates",
):
    for node, state_value in chunk.items():
        print(f"👉 Node: {node}")
        if isinstance(state_value, dict) and state_value.get("messages"):
            state_value["messages"][-1].pretty_print()

## 정리

- **Hierarchical** = Supervisor 를 여러 층으로 쌓아 **팀 → 작업자** 계층을 구성
- **`make_supervisor_node` 팩토리** 로 supervisor 를 층마다 재사용
- **팀 서브그래프를 노드로 감싸** 최상위 supervisor 가 팀을 작업자처럼 호출
- 작업자가 많고 역할이 뚜렷할 때, 평평한 Supervisor 하나보다 계층 분할이 관리에 유리

| 구조 | 핵심 |
|---|---|
| Network | 에이전트끼리 직접 핸드오프 |
| Supervisor | 관리자 1명이 작업자 배분 |
| Hierarchical | 팀(서브 supervisor) 을 상위 supervisor 가 지휘 |

이로써 멀티 에이전트 3대 아키텍처(Network / Supervisor / Hierarchical)를 모두 다뤘다.